In [ ]:
from shapely.geometry import shape, box
from tqdm import tqdm

import requests
import geopandas as gpd

# retrieve philadelphia shapefile from OpenDataPhilly
phl = requests.get(url="https://hub.arcgis.com/api/v3/datasets/405ec3da942d4e20869d4e1449a2be48_0/downloads/data?format=geojson&spatialRefId=4326&where=1%3D1")
phl_json = phl.json()
phl_gdf = gpd.GeoDataFrame.from_features(
    phl_json['features'],
    crs=phl_json['crs']['properties']['name']
).to_crs(crs=4326)

# get philadelphia geometry
phl_geom = phl_gdf.geometry.iloc[0]

# function to intersect with raster bbox
def intersects_phl(item_bbox):
    minx, miny, maxx, maxy = item_bbox
    return box(minx, miny, maxx, maxy).intersects(phl_geom)

In [ ]:
import os
import json
from pystac import Catalog
from pystac.stac_io import DefaultStacIO

# initiate requests session
session_cat = requests.Session()

# start a stac_io session
stac_io = DefaultStacIO()
stac_io.session = session_cat

# load stac catalog
catalog = Catalog.from_file(
    href="https://data.source.coop/nlebovits/phl-aerial-imagery/catalog.json",
    stac_io=stac_io
)

def get_valid_tile_links(catalog: Catalog, year: str, output_dir: str):
    
    collection = catalog.get_child(id=year)

    total = sum(i.rel == "item" for i in collection.links)

    links = {}

    for link in collection.links:
        if link.rel == "item":
            href = link.absolute_href
            key = href.split("/")[-2]
            links[key] = href
    
    print(f"Total Links Found for {year}: {len(links)}")

    os.makedirs(output_dir, exist_ok=True)

    with open(f"{output_dir}/links_{year}.json", "w") as f:
        json.dump(links, f, indent=2)

In [ ]:
# Check if candidate items files already exist
links_2024_path = "./data/identifiers/meta_links/links_2024.json"
links_2025_path = "./data/identifiers/meta_links/links_2025.json"

if not os.path.exists(links_2024_path):
    get_valid_tile_links(catalog=catalog, year="2024", output_dir="./data/identifiers/meta_links")
if not os.path.exists(links_2025_path):
    get_valid_tile_links(catalog=catalog, year="2025", output_dir="./data/identifiers/meta_links")
else:
    print("Link files already exist, skipping download")

with open("./data/identifiers/meta_links/links_2024.json", "r") as f:
    links_2024 = json.load(fp=f)
    print("Loaded 2024 links")

with open("./data/identifiers/meta_links/links_2025.json", "r") as f:
    links_2025 = json.load(fp=f)
    print("Loaded 2025 links")

Link files already exist, skipping download
Loaded 2024 links
Loaded 2025 links


In [ ]:
import os
import json
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed

session_tile = requests.Session()
session_tile.headers.update({"Accept": "application/json"})


def fetch_item(url):
    r = session_tile.get(url, timeout=30)
    r.raise_for_status()
    return r.json()


def get_candidate_items(links: dict, output_dir: str, year: str):

    candidate_items = {}

    with ThreadPoolExecutor(max_workers=2) as ex:
        future_to_key = {
            ex.submit(fetch_item, url): key
            for key, url in links.items()
        }

        for f in tqdm(as_completed(future_to_key), desc="Checking tile overlap", total=len(future_to_key)):
            key = future_to_key[f]
            item = f.result()

            if intersects_phl(item["bbox"]):
                candidate_items[key] = item

    print(f"Found {len(candidate_items)} items for year: {year}")

    os.makedirs(output_dir, exist_ok=True)

    with open(f"{output_dir}/candidate_items_{year}.json", "w") as f:
        json.dump(candidate_items, f, indent=2)

In [ ]:
# Check if candidate items files already exist
candidate_items_2024_path = "./data/identifiers/candidate_items/candidate_items_2024.json"
candidate_items_2025_path = "./data/identifiers/candidate_items/candidate_items_2025.json"

if not os.path.exists(candidate_items_2024_path):
    get_candidate_items(links=links_2024, output_dir="./data/identifiers/candidate_items", year="2024")
if not os.path.exists(candidate_items_2025_path):
    get_candidate_items(links=links_2025, output_dir="./data/identifiers/candidate_items", year="2025")
else:
    print("Candidate items files already exist, skipping download")

with open("./data/identifiers/candidate_items/candidate_items_2024.json", "r") as f:
    candidate_items_2024 = json.load(fp=f)
    print("Loaded 2024 candidate items")

with open("./data/identifiers/candidate_items/candidate_items_2025.json", "r") as f:
    candidate_items_2025 = json.load(fp=f)
    print("Loaded 2025 candidate items")

Candidate items files already exist, skipping download
Loaded 2024 candidate items
Loaded 2025 candidate items


In [ ]:
from urllib.parse import urlparse

def get_candidate_urls(items: dict, output_dir: str, year: str):

    urls = {}
    keys = items.keys()
    
    for key in keys:
        
        item = items[key]
        
        base = "https://data.source.coop/nlebovits/phl-aerial-imagery/"
        collection = item['collection']
        tif = item['assets']['data']['href'].lstrip("./")
        id = item['id']
        href = base + collection + "/" + id + "/" + tif

        def is_valid_url(url):
            try:
                result = urlparse(url)
                return all([result.scheme, result.netloc])
            except ValueError:
                return False
            
        if not is_valid_url(href):
            continue

        urls[id] = href

    if len(urls) < len(items):
        print(f"Some URLs not generated for collection: {collection}")
    else:
        print(f"All URLs generated for collection: {collection}.")

    os.makedirs(output_dir, exist_ok=True)

    with open(f"{output_dir}/data_urls_{year}.json", "w") as f:
        json.dump(urls, f, indent=2)

In [ ]:
# Check if candidate items files already exist
urls_2024_path = "./data/identifiers/data_urls/data_urls_2024.json"
urls_2025_path = "./data/identifiers/data_urls/data_urls_2025.json"

if not os.path.exists(urls_2024_path):
    get_candidate_urls(items=candidate_items_2024, output_dir="./data/identifiers/data_urls", year="2024")
if not os.path.exists(urls_2025_path):
    get_candidate_urls(items=candidate_items_2025, output_dir="./data/identifiers/data_urls", year="2025")
else:
    print("Data url files already exist, skipping download")

with open("./data/identifiers/data_urls/data_urls_2024.json", "r") as f:
    urls_2024 = json.load(fp=f)
    print("Loaded 2024 data urls")

with open("./data/identifiers/data_urls/data_urls_2025.json", "r") as f:
    urls_2025 = json.load(fp=f)
    print("Loaded 2025 data urls")

Data url files already exist, skipping download
Loaded 2024 data urls
Loaded 2025 data urls


In [ ]:
from itertools import islice

urls_2024_select = dict(islice(urls_2024.items(), 32))
urls_2024_select

{'26585e241894n': 'https://data.source.coop/nlebovits/phl-aerial-imagery/2024/26585e241894n/26585E241894N.tif',
 '26611e239254n': 'https://data.source.coop/nlebovits/phl-aerial-imagery/2024/26611e239254n/26611E239254N.tif',
 '26611e241894n': 'https://data.source.coop/nlebovits/phl-aerial-imagery/2024/26611e241894n/26611E241894N.tif',
 '26611e244534n': 'https://data.source.coop/nlebovits/phl-aerial-imagery/2024/26611e244534n/26611E244534N.tif',
 '26638e207574n': 'https://data.source.coop/nlebovits/phl-aerial-imagery/2024/26638e207574n/26638E207574N.tif',
 '26638e210214n': 'https://data.source.coop/nlebovits/phl-aerial-imagery/2024/26638e210214n/26638E210214N.tif',
 '26638e239254n': 'https://data.source.coop/nlebovits/phl-aerial-imagery/2024/26638e239254n/26638E239254N.tif',
 '26638e241894n': 'https://data.source.coop/nlebovits/phl-aerial-imagery/2024/26638e241894n/26638E241894N.tif',
 '26638e247174n': 'https://data.source.coop/nlebovits/phl-aerial-imagery/2024/26638e247174n/26638E247174

In [ ]:
import rioxarray as rxr

def get_downsampled_imgs(urls: dict, output_dir: str):
    
    keys = urls.keys()

    for href_key in tqdm(keys, desc="Retrieving and downsampling specified urls", total=len(keys)):
        data = rxr.open_rasterio(
            urls[href_key],
            masked=True,
            chunks={"band": 1, "y": 1320, "x": 1320}
        )

        data_downsampled = data.coarsen(x=4, y=4, boundary="trim").mean()

        out_path = f"{output_dir}/{href_key}.zarr"
        
        data_downsampled.to_dataset(name="data").to_zarr(
            out_path,
            mode="w",
            zarr_format=2
        )

In [ ]:
import rioxarray as rxr
from dask import compute, delayed
from dask.distributed import Client

def process_one(url, out_path):
    data = rxr.open_rasterio(
        url,
        masked=True,
        chunks={"band": 1, "y": 1320, "x": 1320}
    )

    data_downsampled = data.coarsen(x=4, y=4, boundary="trim").mean()

    return data_downsampled.to_dataset(name="data").to_zarr(
        out_path,
        mode="w",
        zarr_format=2,
        compute=False
    )

def get_downsampled_imgs_parallel(urls: dict, output_dir: str):
    writes = [
        process_one(urls[k], f"{output_dir}/{k}.zarr")
        for k in urls
    ]

    with Client(n_workers=16, threads_per_worker=1) as client:
        compute(*writes)

In [ ]:
get_downsampled_imgs_parallel(urls=urls_2024, output_dir="./data/imagery/2024")
get_downsampled_imgs_parallel(urls=urls_2025, output_dir="./data/imagery/2025")

c:\Users\Henry\Desktop\MUSA6500_Desktop\musa6500-finalproject\.venv\Lib\site-packages\distributed\client.py:3398: UserWarning: Sending large graph of size 167.79 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
2026-04-19 02:54:43,607 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killing
2026-04-19 02:54:43,660 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killing
2026-04-19 02:54:43,662 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killing
2026-04-19 02:54:43,664 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killing
2026-04-19 02:54:43,666 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killin